# EvalForge: Complete Project Walkthrough

This notebook turns the multi-file harness into one guided experiment. It audits the 28-task benchmark, validates a real task, runs one deterministic repair through both test gates, and analyses the 60 saved three-condition apparatus runs.

**Integrity boundary:** the saved 60/60 result validates the harness with reference scripts. It is not LLM coding performance.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'src' / 'evalforge').exists(), 'Launch this notebook from the evalforge project directory.'
sys.path.insert(0, str(ROOT / 'src'))
(ROOT / 'artifacts' / 'notebook').mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])


Project root: .
Python: 3.12.14


## 1. Benchmark dataset audit


In [2]:
catalog = [json.loads(line) for line in (ROOT/'data/benchmark_catalog.jsonl').read_text().splitlines()]
catalog_df = pd.DataFrame(catalog)
print('Tasks:', len(catalog_df))
print(catalog_df.groupby(['suite','category']).size().to_string())
catalog_df.head()


Tasks: 28
suite        category                      
development  device_dtype_mismatch             1
             numerical_stability               1
             numpy_shape_broadcasting          1
             pandas_data_leakage               1
             pytorch_training_loop             1
             random_seed_handling              1
             regression_metrics                1
             scientific_unit_conversion        1
evaluation   bad_tensor_dimensions             1
             classification_metrics            2
             data_split_mistakes               1
             device_dtype_mismatch             2
             incorrect_gradient_handling       2
             numerical_stability               2
             numpy_shape_broadcasting          2
             pandas_data_leakage               1
             pytorch_training_loop             2
             random_seed_handling              1
             regression_metrics                1
             re

## 2. Validate one buggy/reference task pair


In [3]:
from evalforge.benchmark import load_task, validate_task_behavior
task_path = ROOT/'benchmarks/development/task_001'
validation = validate_task_behavior(load_task(task_path))
print(json.dumps(validation.model_dump(mode='json'), indent=2))
assert validation.valid


{
  "task_id": "numpy_broadcast_001",
  "valid": true,
  "buggy_state_failed": true,
  "reference_patch_applied": true,
  "repaired_state_passed": true,
  "buggy_verification": {
    "command": [
      "python",
      "-m",
      "pytest",
      "-q",
      "tests"
    ],
    "exit_code": 1,
    "passed": false,
    "output": "FF                                                                       [100%]\n=================================== FAILURES ===================================\n____________________ test_standardizes_each_feature_column _____________________\n\n    def test_standardizes_each_feature_column() -> None:\n        values = np.array(\n            [\n                [1.0, 10.0],\n                [2.0, 20.0],\n                [3.0, 30.0],\n                [4.0, 40.0],\n            ]\n        )\n    \n>       transformed = zscore_features(values)\n                      ^^^^^^^^^^^^^^^^^^^^^^^\n\ntests/test_normalization.py:17: \n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _

## 3. Run the verifier-first state machine


In [4]:
from evalforge.benchmark import read_script
from evalforge.models import AgentDecision, Budget, Condition
from evalforge.orchestrator import Orchestrator
from evalforge.providers.mock import ScriptedProvider

task = load_task(task_path)
decisions = [AgentDecision.model_validate(item) for item in read_script(task_path/'scripted_actions.json')]
summary = Orchestrator(
    task=task,
    provider=ScriptedProvider(decisions),
    run_root=ROOT/'artifacts/notebook/runs',
    budget=Budget(max_iterations=8, max_tool_calls=40),
    condition=Condition.SCRIPTED,
).run()
print(json.dumps(summary.model_dump(mode='json'), indent=2))
assert summary.resolved and summary.evaluator_verification.passed


{
  "run_id": "numpy_broadcast_001-f593b2a302",
  "task_id": "numpy_broadcast_001",
  "condition": "scripted",
  "started_at": "2026-09-11T18:20:06.849695Z",
  "finished_at": "2026-09-11T18:20:07.910275Z",
  "termination_reason": "tests_passed",
  "resolved": true,
  "visible_verification": {
    "command": [
      "python",
      "-m",
      "pytest",
      "-q",
      "tests"
    ],
    "exit_code": 0,
    "passed": true,
    "output": "..                                                                       [100%]\n2 passed in 0.09s\n",
    "runtime_seconds": 0.3391338519999181,
    "tests_collected": 2,
    "tests_passed": 2,
    "tests_failed": null,
    "environment_error": false
  },
  "evaluator_verification": {
    "command": [
      "python",
      "-m",
      "pytest",
      "-q",
      "tests",
      ".evalforge_evaluator_tests"
    ],
    "exit_code": 0,
    "passed": true,
    "output": ".....                                                                    [100%]\n5 pa

## 4. Analyse the saved three-condition apparatus exercise


In [5]:
from evalforge.evaluation import load_summaries, aggregate
saved = load_summaries(ROOT/'experiments/artifacts/scripted-validation')
rows = aggregate(saved)
results = pd.DataFrame(rows)
print('Saved runs:', len(saved), '| unique tasks:', len({item.task_id for item in saved}))
results[['condition','runs','unique_tasks','task_resolution_rate','average_iterations','average_runtime_seconds']]


Saved runs: 60 | unique tasks: 20
         condition  runs  ...  average_iterations  average_runtime_seconds
0         baseline    20  ...                 2.0                 3.175716
1           memory    20  ...                 2.0                 3.195185
2  memory_verifier    20  ...                 2.0                 3.143039

[3 rows x 6 columns]


In [6]:
figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(results['condition'], results['task_resolution_rate'], color='#315c8c')
axes[0].set_ylim(0, 1.05); axes[0].set_title('Apparatus resolution rate'); axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(results['condition'], results['average_runtime_seconds'], color='#35a36f')
axes[1].set_title('Average runtime'); axes[1].tick_params(axis='x', rotation=20)
figure.tight_layout(); figure.savefig(ROOT/'artifacts/notebook/evalforge_ablation.png', dpi=160)
plt.show()


## 5. What to run next

- Credential-free validation: `evalforge validate-suite benchmarks/evaluation --execute`
- One deterministic demo: `evalforge run-scripted benchmarks/development/task_001`
- Real model run: install `.[llm,benchmark]`, configure the provider, then use `evalforge run-task ...`.
- Never place the scripted 100% result on a CV as LLM performance. The defensible numbers are the engineered benchmark size, test suite, state-machine features, and number of audited apparatus runs.
